# Feature Engineering

This notebook focuses on transforming telemetry data into actionable performance indicators. By calculating metrics for speed, braking, throttle application, and gear usage, the process creates a structured dataset that quantifies driving behavior, enabling a deeper understanding of track management.

## Setup and Dependencies

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [1]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.config import *
from src.load_save_data import *
from src.preprocess_data import *
from src.explore_data import *
from src.feature_engineering import *

## Load Processed Data

Fetch cleaned and interpolated telemetry data for both teammates. Loading pre-prepared data ensures that feature calculations are performed on standardized, high-quality inputs.

In [2]:
session = fastf1.get_session(YEAR, GRAND_PRIX, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, TEAM)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(YEAR, GRAND_PRIX, segment, driver_1, driver_2)
except Exception as e:
    print(f"Processed data for {YEAR} {GRAND_PRIX} {driver_1} and {driver_2} not found: {e}")

req         WARNING 	DEFAULT CACHE ENABLED! (99.92 MB) C:\Users\joseg\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '6', '44', '23', '87', '10', '55', '14', '30', '22', '27', '5', '31', '7', '18']



Processed data for 2025 Japanese Grand Prix ALO and STR found in cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1


## Compute Speed Segments

Compute speed-related performance metrics such as: entry speed, minimum speed and exit speed throughout the corners. This analysis examines speed profiles relative to track turns to identify differences in corner entry and exit speeds, providing insights into raw pace and efficiency.

In [3]:
speed_segments_1 = analyze_speed_segments(lap_1, turns)
speed_segments_2 = analyze_speed_segments(lap_2, turns)

print(f"Speed segments for {driver_1}:")
display(speed_segments_1)
print(f"Speed segments for {driver_2}:")
display(speed_segments_2)

Speed segments for ALO:


,Segment,ExitSpeed,MinimumSpeed,MeanSpeed,StdSpeed,EntrySpeed
0,0,218.830,226.000,271.954,32.354,316.850
1,1,188.211,175.000,186.742,10.810,218.830
2,2,254.000,239.046,247.756,6.692,237.207
3,3,229.825,225.316,233.834,7.681,249.991
4,4,230.018,228.000,233.839,5.565,242.827
5,5,193.286,192.259,209.650,14.486,239.000
6,6,263.927,248.334,254.332,5.221,247.558
7,7,253.581,255.966,269.242,9.924,287.242
8,8,156.727,131.648,155.865,23.508,222.547
9,9,244.286,260.651,262.901,0.931,259.497


Speed segments for STR:


,Segment,ExitSpeed,MinimumSpeed,MeanSpeed,StdSpeed,EntrySpeed
0,0,215.922,221.783,276.522,31.762,310.668
1,1,183.305,171.071,181.966,10.163,215.922
2,2,247.529,238.790,247.747,4.396,235.215
3,3,228.708,227.000,237.155,8.196,246.000
4,4,225.454,218.000,226.325,8.508,241.535
5,5,192.897,194.030,212.162,13.322,235.749
6,6,259.000,240.479,249.359,5.853,238.936
7,7,247.620,246.461,262.404,15.026,287.000
8,8,144.203,133.547,163.638,29.818,240.580
9,9,246.919,257.988,261.655,1.677,256.377


## Compute Braking Zones

Identify and analyze braking zones throughout the lap. This section evaluates how each driver approaches braking points—specifically focusing on the start and end of braking events—to highlight differences in commitment and corner entry style registering information such as: braking duration, braking distance and braking point.

In [4]:
braking_segments_1 = analyze_braking_segments(lap_1, turns)
braking_segments_2 = analyze_braking_segments(lap_2, turns)

print(f"Braking segments for {driver_1}:")
display(braking_segments_1)
print(f"Braking segments for {driver_2}:")
display(braking_segments_2)

Braking segments for ALO:


,Segment,BrakingPoint,BrakingDistance,BrakingDuration
0,0,31.944321,94.451661,1.403598
1,1,0.000000,0.000000,0.000000
2,2,0.000000,0.000000,0.000000
3,3,0.000000,0.000000,0.000000
4,4,0.000000,0.000000,0.000000
5,5,53.215447,42.932573,0.710531
6,6,0.000000,0.000000,0.000000
7,7,33.348841,51.519088,0.718115
8,8,68.549995,68.692117,1.414791
9,9,0.000000,0.000000,0.000000


Braking segments for STR:


,Segment,BrakingPoint,BrakingDistance,BrakingDuration
0,0,31.944321,103.038175,1.571767
1,1,0.000000,0.000000,0.000000
2,2,0.000000,0.000000,0.000000
3,3,46.566067,42.932573,0.643467
4,4,33.166824,34.346058,0.559168
5,5,44.628932,8.586515,0.137719
6,6,0.000000,0.000000,0.000000
7,7,33.348841,34.346058,0.476000
8,8,68.549995,77.278632,1.612747
9,9,0.000000,0.000000,0.000000


## Compute Throttle Segments

Calculate throttle usage segments. This analysis determines when and how aggressively each driver applies the throttle coming out of corners, serving as a critical indicator of traction management and power deployment. Each segment registers: mean throttle, throttle standard deviation and full throttle percentage. 

In [5]:
throttle_segments_1 = analyze_throttle_segments(lap_1, turns)
throttle_segments_2 = analyze_throttle_segments(lap_2, turns)

print(f"Throttle segments for {driver_1}:")
display(throttle_segments_1)
print(f"Throttle segments for {driver_2}:")
display(throttle_segments_2)

Throttle segments for ALO:


,Segment,MeanThrottle,StdThrottle,FullThrottlePercentage
0,0,92.647100,21.778178,86.904762
1,1,5.478588,9.032246,0.000000
2,2,89.058511,22.484614,77.777778
3,3,69.351708,18.929261,18.750000
4,4,72.245615,24.165788,36.842105
5,5,51.720823,26.911346,5.263158
6,6,86.000652,28.211506,72.972973
7,7,93.091439,18.227434,87.755102
8,8,50.805418,40.363838,17.647059
9,9,91.314393,22.835585,84.615385


Throttle segments for STR:


,Segment,MeanThrottle,StdThrottle,FullThrottlePercentage
0,0,98.293451,8.509816,97.619048
1,1,2.272783,5.640167,0.000000
2,2,92.352880,14.044979,77.777778
3,3,79.305871,17.203975,25.000000
4,4,67.921587,30.576488,26.315789
5,5,60.225034,43.590728,42.105263
6,6,82.506505,27.461524,62.162162
7,7,91.615371,25.066260,89.795918
8,8,55.102156,46.488676,35.294118
9,9,87.046364,28.912688,79.487179


## Compute Gear Shifts Segments

Examine gear shift patterns across the lap registering: number of shifts, lowest gear, highest gear, minimum rpm, maximum rpm, mean rpn and rpm standard deviation. Comparing gear selection at different track segments reveals variations in driving technique, engine management, and how torque and traction are balanced.

In [6]:
gear_shifts_segments_1 = analyze_gear_shifts_segments(lap_1, turns)
gear_shifts_segments_2 = analyze_gear_shifts_segments(lap_2, turns)

print(f"Gear shifts segments for {driver_1}:")
display(gear_shifts_segments_1)
print(f"Gear shifts segments for {driver_2}:")
display(gear_shifts_segments_2)

Gear shifts segments for ALO:


,Segment,NumberOfShifts,LowestGear,HighestGear,MinimumRPM,MaximumRPM,MeanRPM,StdRPM
0,0,1,7,8,10342,11836,11178.976190,332.855661
1,1,3,5,8,9816,10659,10285.230769,253.066577
2,2,1,5,6,9541,11838,10901.555556,739.657838
3,3,0,6,6,10923,12004,11676.125000,369.464094
4,4,0,6,6,10766,11357,11071.789474,212.931021
5,5,1,5,6,10634,11467,11033.315789,205.927186
6,6,2,5,7,10356,11943,11074.621622,453.344017
7,7,0,7,7,10586,11776,11374.204082,365.357586
8,8,1,4,7,9756,10737,10343.647059,266.318639
9,9,5,3,7,9794,11655,10985.538462,433.235492


Gear shifts segments for STR:


,Segment,NumberOfShifts,LowestGear,HighestGear,MinimumRPM,MaximumRPM,MeanRPM,StdRPM
0,0,1,7,8,10244,11668,11116.130952,319.209633
1,1,3,5,8,9634,11120,10277.615385,461.640108
2,2,1,5,6,9402,11837,10839.222222,745.754669
3,3,0,6,6,11166,11914,11662.187500,233.172102
4,4,0,6,6,10444,11467,10979.789474,281.442431
5,5,1,5,6,10433,11503,10950.842105,305.265142
6,6,1,5,6,10497,11603,11086.135135,318.456360
7,7,2,6,7,10359,11755,11260.122449,429.054126
8,8,1,5,6,10071,12135,11343.764706,635.719821
9,9,4,4,7,8411,11464,10527.358974,841.791526


## Compute Segment Times

In [7]:
times_segments_1 = analyze_time_segments(lap_1, turns)
times_segments_2 = analyze_time_segments(lap_2, turns)

print(f"Times segments for {driver_1}:")
display(times_segments_1)
print(f"Times segments for {driver_2}:")
display(times_segments_2)

Times segments for ALO:


,Segment,StartDistance,EndDistance,SegmentTime
0,0,0.000000,718.865490,8.318758
1,1,718.865490,830.361033,1.748213
2,2,830.361033,1063.112431,3.791294
3,3,1063.112431,1197.159026,1.917242
4,4,1197.159026,1364.076590,2.377794
5,5,1364.076590,1521.509446,2.454693
6,6,1521.509446,1841.732500,4.926145
7,7,1841.732500,2265.842642,5.348176
8,8,2265.842642,2412.668486,2.303449
9,9,2412.668486,2743.644082,5.639452


Times segments for STR:


,Segment,StartDistance,EndDistance,SegmentTime
0,0,0.000000,718.865490,8.393278
1,1,718.865490,830.361033,1.827904
2,2,830.361033,1063.112431,3.792102
3,3,1063.112431,1197.159026,1.895474
4,4,1197.159026,1364.076590,2.404166
5,5,1364.076590,1521.509446,2.476878
6,6,1521.509446,1841.732500,5.019166
7,7,1841.732500,2265.842642,5.431299
8,8,2265.842642,2412.668486,2.299278
9,9,2412.668486,2743.644082,5.683200


## Cache Processed Data

Integrate the newly calculated metrics into the driver data structures and save the augmented dataset to the cache. This ensures that extracted features are persistent and readily available for subsequent performance comparisons and visualization steps.

In [8]:
segments_features_1 = build_segments_dataframe(times_segments_1, speed_segments_1, braking_segments_1, throttle_segments_1, gear_shifts_segments_1)
segments_features_2 = build_segments_dataframe(times_segments_2, speed_segments_2, braking_segments_2, throttle_segments_2, gear_shifts_segments_2)

lap_1["Segments"] = segments_features_1
lap_2["Segments"] = segments_features_2

save_dataframe(lap_1, "Segments", YEAR, GRAND_PRIX)
save_dataframe(lap_2, "Segments", YEAR, GRAND_PRIX)

try:
    save_cache(YEAR, GRAND_PRIX, segment, lap_1, lap_2)
except Exception as e:
    print(f"Failed to save lap cache: {e}")

Saving payload to E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_japanese_grand_prix_Q1_ALO.pkl
Saved: E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_japanese_grand_prix_Q1_ALO.pkl
Saving payload to E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_japanese_grand_prix_Q1_STR.pkl
Saved: E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_japanese_grand_prix_Q1_STR.pkl

Processed data for 2025 Japanese Grand Prix saved to cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1
